# TinyLlama quantized (GGUF) local exploration

This notebook downloads and runs a quantized TinyLlama model using `llama-cpp-python` (great for local Mac experimentation).

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import hf_hub_download, whoami
from llama_cpp import Llama


## 1) Load `.env` and optional Hugging Face token

Public GGUF files can be downloaded without a token, but `HF_TOKEN` is still useful for higher rate limits.

In [ ]:
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env')
HF_TOKEN = (os.getenv('HF_TOKEN') or '').strip() or None

if HF_TOKEN:
    try:
        user = whoami(token=HF_TOKEN)
        print(f"HF token loaded for user: {user.get('name')}")
    except Exception as e:
        print(f"Token exists but validation failed: {e}")
else:
    print('No HF_TOKEN found in .env. Proceeding with public access.')


## 2) Choose quantized TinyLlama variant

`Q4_K_M` is a good quality/speed/size balance for local testing.

In [ ]:
GGUF_REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'
GGUF_FILE = 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf'
print('Repo:', GGUF_REPO)
print('File:', GGUF_FILE)


## 3) Configure cache and download GGUF

If your default disk is low, set `HF_HOME` to another drive path before running this cell.

In [ ]:
HF_HOME = os.getenv('HF_HOME') or str(Path.home() / '.cache' / 'huggingface')
os.environ['HF_HOME'] = HF_HOME

cache_path = Path(HF_HOME)
cache_path.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(cache_path).free / (1024 ** 3)
print(f'HF cache path: {cache_path}')
print(f'Free space: {free_gb:.2f} GB')

if free_gb < 2.0:
    raise RuntimeError(
        'Low disk space for GGUF download. Need about 2+ GB free. '
        'Free space or set HF_HOME to a larger disk path.'
    )

model_path = hf_hub_download(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    token=HF_TOKEN,
)

print('Downloaded/cached GGUF at:')
print(model_path)


## 4) Load quantized model with llama.cpp

On Apple Silicon, Metal acceleration is used when available by setting `n_gpu_layers=-1`.

In [ ]:
is_macos = platform.system() == 'Darwin'
n_gpu_layers = -1 if is_macos else 0

llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=max((os.cpu_count() or 4) - 1, 2),
    n_gpu_layers=n_gpu_layers,
    verbose=False,
)

print('Model loaded. n_gpu_layers =', n_gpu_layers)


## 5) Run a quick prompt

In [ ]:
prompt = 'Write 3 practical tips for learning open-source LLMs quickly.'

chat_prompt = (
    '<|system|>\nYou are a concise, helpful AI assistant.\n'
    '<|user|>\n' + prompt + '\n'
    '<|assistant|>\n'
)

out = llm(
    chat_prompt,
    max_tokens=180,
    temperature=0.7,
    top_p=0.9,
    stop=['<|user|>', '</s>'],
)

print(out['choices'][0]['text'].strip())
